<a href="https://colab.research.google.com/github/Datdeptrai912005/DemoGit/blob/main/Tutorial_NLTK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![PALS0039 Logo](https://www.phon.ucl.ac.uk/courses/pals0039/images/pals0039logo.png)](https://www.phon.ucl.ac.uk/courses/pals0039/)

# Tutorial Introduction to NLTK toolkit

This notebook adapted from the [NLTK Book](http://www.nltk.org/book/ch01.html) Chapter 1.

(a) Import the NLTK module and download the text resources needed for the examples.

In [ ]:
# 1. Cài đặt thư viện cần thiết
!pip install -q transformers accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 2. Tải mô hình và tokenizer (bản 1.5B rất nhẹ cho Colab)
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. Thử nghiệm đặt câu hỏi
prompt = "Viết một bài thơ ngắn về thành phố Quy Nhơn."
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

model_inputs = tokenizer([text], return_tensors="pt").to(device)

generated_ids = model.generate(model_inputs.input_ids, max_new_tokens=512)
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print(response.split("assistant")[-1])



(b) Take a sentence and tokenize into words. Then apply a part-of-speech tagger.

In [ ]:
# Cài đặt bản unsloth để chạy siêu nhanh trên Colab T4
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


(c) From the tagged words, identify the proper names.

In [ ]:
!pip install newspaper3k
!pip install newspaper3k lxml_html_clean
from newspaper import Article

url = "https://vov.vn/the-gioi/dien-bien-chinh-tinh-hinh-chien-su-nga-ukraine-ngay-123-post1274891.vov"

def doc_bao_tu_link(url):
    article = Article(url)
    article.download()
    article.parse()
    return article.text

# Lấy nội dung
noi_dung_tu_link = doc_bao_tu_link(url)

# Đưa vào AI phân tích tin giả
phan_tich = check_fake_news(noi_dung_tu_link)
print(phan_tich)


(d) get texts for corpus analysis


In [ ]:
def check_fake_news(news_content):
    prompt = f"""Bạn là một chuyên gia kiểm chứng tin tức (fact-checker) chuyên sâu về tình hình chiến sự.
Hãy phân tích nội dung dưới đây và đưa ra kết luận:
1. Đánh giá: (THẬT / GIẢ / TIN ĐỒN)
2. Phân tích chi tiết: (Kiểm tra các địa danh, tên riêng và logic của thông tin)
3. Độ tin cậy: (0-100%)

Nội dung tin: {news_content}
Kết luận:"""

    messages = [{"role": "user", "content": prompt}]

    # Chuyển đổi prompt sang định dạng mô hình hiểu được
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    # Tạo câu trả lời
    outputs = model.generate(input_ids=inputs, max_new_tokens=1024, temperature=0.1)
    response = tokenizer.batch_decode(outputs)

    # Trả về kết quả sạch
    return response[0].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "")

(e) generate a key-word in context concordance

In [ ]:
text1.concordance("monstrous")

(f) find words with similar concordance to a given word

In [ ]:
print(text1)
text1.similar("monstrous")
print(text2)
text2.similar("monstrous")


(g) find contexts which are similar for the given words

In [ ]:
text2.common_contexts(["monstrous", "very"])

(h) plot where in the text certain words appear

In [ ]:
text4.dispersion_plot(["citizens", "democracy", "freedom", "duties", "America"])

(i) print the identity of a text, the length of the text and its vocabulary

In [ ]:
print(text3)
print(len(text3))
print(sorted(set(text3)))

(j) print some statistics of word occurrence in the text

In [ ]:
def lexical_diversity(text):
  return len(set(text)) / len(text)
def percentage(count, total):
  return 100 * count / total

print(lexical_diversity(text3))
print(lexical_diversity(text5))
print(percentage(text4.count('a'), len(text4)))
